# Assignment 2 — Agents and Prebuilt Middleware

**Domain for this assignment: NovaBank — a personal banking assistant.** A different scenario
from CineBot/GreenPlate/TripMate, used throughout your notebooks — the goal is proving you
understand the *concepts*, not that you can copy-paste code you've already seen with renamed
variables.

**Scope, explicitly:** this assignment covers everything through **Agents** and **prebuilt
middleware** — `create_agent`, `thread_id`, `context_schema`, and the built-in middleware
library (`HumanInTheLoopMiddleware`, `PIIMiddleware`, retry/limit/fallback middleware, and the
rest). **Writing your own custom middleware is intentionally not required anywhere in this
assignment** — every exercise here uses only built-in, pre-existing middleware classes.

**Structure:** Part A is conceptual, Part B is coding, both easier to harder. Part C is a single
capstone. Attempt sections in order.

**Before you start:** confirm your environment is set up and you can run a basic `create_agent`
call successfully.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Missing OPENAI_API_KEY -- check your .env file or Colab Secrets"

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
model = init_chat_model("openai:gpt-5-mini")
print("Environment ready.")


---
# Part A — Conceptual Questions

## A1. Agents (Easy–Medium)

1. Describe the agent loop, mechanically, in your own words — what happens between a user's
   message going in and a final answer coming out, when a tool call is involved?

2. `thread_id` alone doesn't persist a conversation. What else is required, and what specifically
   breaks if you forget it?

3. Precisely distinguish `thread_id` from `context`. Give one example of data that belongs in
   each, and explain why swapping them would cause a real bug.

4. Name the three practical `stream_mode` options covered in this course and what each is best
   suited for.

5. Why does naming an agent (`name=`) cost nothing now but matter later?

## A2. Prebuilt Middleware Core (Medium)

6. State the exact rule for how `before_*`, `after_*`, and `wrap_*` hooks order themselves when
   multiple middleware are combined. Why are `before_*` and `after_*` different from each other?

7. `HumanInTheLoopMiddleware` supports four decisions. For a NovaBank `transfer_funds` tool,
   describe a realistic situation where each of the four would actually be the right choice.

8. `SummarizationMiddleware` and `ContextEditingMiddleware` both manage growing context. If
   NovaBank's agent has a handful of very simple tools but very long, chatty conversations, which
   is the better fit, and why?

9. This framework has three genuinely distinct "retry" mechanisms. For each, give one NovaBank
   scenario where that SPECIFIC one (and not the other two) is the right tool.

10. `ToolCallLimitMiddleware` and `ModelCallLimitMiddleware` are both "limits," but limit
    different things. Give a NovaBank scenario where you'd want a strict `ToolCallLimitMiddleware`
    on one specific tool, but a looser `ModelCallLimitMiddleware` overall.

## A3. Prebuilt Middleware, Advanced (Hard)

11. `ToolErrorMiddleware` and `ToolRetryMiddleware` are commonly used together. Explain the
    correct composition (which goes where in the `middleware` list, and why), and what happens
    if you get the order backwards.

12. `PIIMiddleware` has three independent "apply to" flags. Design a PII policy for NovaBank
    covering account numbers: which flags would you set to `True`, and what real scenario does
    each one protect against?

13. Explain the real, documented discrepancy involving `LLMToolEmulator`'s `model` parameter.
    Why is explicitly setting `model=` a good habit for any middleware that accepts its own
    separate model, not just this one?

14. NovaBank's agent has grown to 15 tools. Compare two different fixes: `LLMToolSelectorMiddleware`
    versus `ProviderToolSearchMiddleware`. What's the real difference in HOW each one filters
    tools, and what constraint limits when you can use the second one?

15. **Design question (open-ended):** sketch a complete prebuilt-middleware stack (using ONLY
    built-in middleware — no custom code) for a production NovaBank agent that can check
    balances, transfer funds, and answer general questions. List which middleware you'd include,
    in what order, and justify each choice. There's no single correct answer.


---
# Part B — Coding Exercises

All exercises use **NovaBank**, a personal banking assistant. Every exercise here uses only
**built-in** middleware — no `@before_model`, no subclassing `AgentMiddleware`, anywhere in this
section.

## B1. Easy

**B1.1 — A basic agent with real memory.** Write two tools: `check_balance` (takes an
`account_id: str`, returns a fake balance) and `get_interest_rate` (no arguments, returns a fake
rate). Build a `create_agent` with a checkpointer, and prove conversation memory works across two
separate `.invoke()` calls on the same `thread_id` (e.g. tell it your name in call 1, ask for it
back in call 2).


In [ ]:
# Your solution for B1.1


**B1.2 — Per-run context.** Define a `context_schema` (a dataclass) carrying `account_tier`
(e.g. `"standard"` or `"premium"`). Write a tool that reads `runtime.context.account_tier` and
returns a different greeting depending on the tier. Invoke the agent twice with two different
context values and show the difference.


In [ ]:
# Your solution for B1.2


## B2. Medium

**B2.1 — Guard a consequential action with all four HITL decisions.** Write a `transfer_funds`
tool (takes `to_account: str`, `amount: float`). Guard it with `HumanInTheLoopMiddleware`
allowing all four decisions. Trigger the interrupt, then resume it with an `edit` decision that
changes the amount before it goes through. Print the final result and confirm the edited amount
was actually used.


In [ ]:
# Your solution for B2.1


**B2.2 — Protect account numbers with a custom detector.** NovaBank account numbers follow the
format `NB-` followed by 8 digits. Write a custom detector function matching this pattern and
attach it via `PIIMiddleware` with `strategy="mask"`. Prove it redacts an account number
appearing in a test message.


In [ ]:
# Your solution for B2.2


**B2.3 — Retry a flaky balance-check API.** Write a tool `check_live_balance` that randomly
raises a `ConnectionError` about half the time (simulating a real banking API). Attach
`ToolRetryMiddleware` with your own retry/backoff settings, and run it enough times to observe
both a case that succeeds on the first try and one that needs a retry (print inside the tool to
show each attempt).


In [ ]:
# Your solution for B2.3


## B3. Hard

**B3.1 — Compose three limit/context middleware together.** Combine `SummarizationMiddleware`,
`ToolCallLimitMiddleware` (tight limit on `transfer_funds` specifically, looser overall), and
`ModelCallLimitMiddleware` on one agent. Explain in a markdown cell what real problem each one is
solving simultaneously, then demonstrate the agent still working normally within those limits.


In [ ]:
# Your solution for B3.1


**B3.2 — Tool selection at scale.** Give NovaBank at least 6 tools (a mix of real and stubbed —
balance checks, transfers, loan info, branch locator, interest rates, support ticket creation).
Attach `LLMToolSelectorMiddleware` with `max_tools=3` and `always_include` set to whichever tool
should never be filtered out. Ask a question and confirm only the expected subset of tools was
available for that specific call.


In [ ]:
# Your solution for B3.2


**B3.3 — Compose `ToolErrorMiddleware` and `ToolRetryMiddleware` correctly.** Write a tool that
raises a `ValueError` on bad input (not a transient failure — always fails on that specific bad
input, no matter how many times you retry it). Combine `ToolRetryMiddleware` (correctly
configured with `on_failure="error"`) and `ToolErrorMiddleware`, in the correct order, so the
final error message is safe and controlled rather than a raw exception. Prove the final message
never leaks the raw Python exception text.


In [ ]:
# Your solution for B3.3


---
# Part C — Capstone Challenge

Build a single, complete `create_agent` for NovaBank combining **at least six** of the following
(your choice which six, but justify your choices in a markdown cell before your code) — **using
only built-in middleware and agent features, no custom middleware authoring**:

- A `response_format` schema for a structured banking request (with at least one real constraint)
- At least three real tools (balance check, transfer, and one more of your choice)
- `HumanInTheLoopMiddleware` guarding the transfer tool
- `PIIMiddleware` protecting at least one PII type
- A retry middleware (tool or model level) protecting against transient failures
- A call-limit middleware protecting against runaway usage
- `context_schema` carrying at least one piece of per-run data a tool reads
- Short-term memory via a checkpointer and consistent `thread_id`

**Requirements:**
1. A markdown cell explaining your design choices before the code.
2. The complete, runnable code.
3. At least two `.invoke()` calls demonstrating the system actually working.
4. A short markdown reflection (3–5 sentences): which ONE piece of this stack would you add
   NEXT, once you're allowed to write custom middleware, and what would it do?

This is intentionally open-ended — the goal is combining prebuilt pieces coherently, not
matching a hidden architecture.


*Design explanation goes here (before your code):*

In [ ]:
# Your capstone solution


*Your reflection goes here:*